# Building an LLM-RAG System for Patient Q&A

In this lab, we will build a Retrieval-Augmented Generation (RAG) system using a large language model (LLM) for patient Q&A. The system will be based on the Patient Information section of the NHS.UK website, which explains diseases, their symptoms and medications. The dataset for this task was created synthetically with the help of ChatGPT, as detailed in this [article](https://aiforhealthcare.substack.com/p/a-large-language-model-for-healthcare).

Load the Q&A dataset (provided with the lab) and perform Exploratory Data Analysis (EDA). Check the distribution of diagnoses, lengths of questions and answers in words.

**Connect to a GPU runtime environment if you have access to one (Runtime -> Change runtime type). Otherwise, set the variable below to True to use API calls.**

In [67]:
use_api = False

In [68]:
import pandas as pd

data_path = "patient_qa.csv"
df = pd.read_csv(data_path)

# disease distribution
diagnosis_counts = df['disease'].value_counts()

print("Disease distribution")
print(diagnosis_counts)

Disease distribution
disease
Bronchiolitis                          2
Laryngitis                             2
Multiple sclerosis                     2
Pneumonia                              2
Tonsillitis                            2
acanthosis nigricans                   2
achalasia                              2
acne                                   2
acoustic neuroma                       2
acute cholecystitis                    2
acute kidney injury                    2
acute respiratory distress syndrome    2
adenoidectomy                          2
Name: count, dtype: int64


In [69]:
df["question_length"] = df["question"].apply(lambda x: len(str(x).split()))
df["answer_length"] = df["answer"].apply(lambda x: len(str(x).split()))

print("\n Question word length stats:")
print(df["question_length"].describe())

print("\n Answer word length stats:")
print(df["answer_length"].describe())


 Question word length stats:
count    26.000000
mean      6.846154
std       2.361225
min       3.000000
25%       5.250000
50%       7.000000
75%       8.750000
max      10.000000
Name: question_length, dtype: float64

 Answer word length stats:
count    26.000000
mean     43.423077
std      22.480521
min      14.000000
25%      27.000000
50%      36.000000
75%      58.250000
max      97.000000
Name: answer_length, dtype: float64


## Patient Q&A with the MediPhi LLM

Now, let's move on to the main part of the lab where we will load a pre-trained model from [HuggingFace](https://huggingface.co) and set up a text generation pipeline. As you already know, HuggingFace provides a library of pre-trained models. One such model is the [MediPhi-Guidelines](https://huggingface.co/microsoft/MediPhi-Guidelines). MediPhi adapts the Phi-3.5 model (3.8B parameters, https://arxiv.org/abs/2404.14219) to healthcare by combining knowledge from multiple specialised medical experts and clinical instruction datasets into a single medical LLM.

**Note**: if you are using CPU and API calls you will be using [Qwen3](https://arxiv.org/abs/2505.09388).

The following code below shows how to load the MediPhi-Guidelines model and create a text generation pipeline.

In [70]:
import os
if use_api:
  # temporary token, replace with your token in follow-up work
  REPLICATE_API_TOKEN  = "r8_Mx6AJvw65Y4K2kJSQwO3I8R726fOCS847DV5Z"
  os.environ["REPLICATE_API_TOKEN"] = REPLICATE_API_TOKEN

In [71]:
!pip install replicate

In [72]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import set_seed
import torch
import re
import numpy as np
import random


# set all random seeds
SEED = 42
set_seed(SEED)
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
  torch.cuda.manual_seed_all(SEED)


if use_api:

    import replicate
    from replicate.client import Client

    replicate = Client(api_token=REPLICATE_API_TOKEN)


    def pipe(prompt, max_new_tokens=100, temperature=0, return_full_text=False, do_sample=False, clean_up_tokenization_spaces=False):
        output = replicate.run(
            "qwen/qwen3-235b-a22b-instruct-2507",
            input={
                "prompt": ''.join(item['content'] for item in prompt),
                "max_tokens": max_new_tokens,
                "temperature": temperature
            }
        )
        return [{"generated_text": "".join(output)}]

else:

 model_id = "microsoft/MediPhi-Guidelines"
 device = "cuda" if torch.cuda.is_available() else "cpu"
 tokenizer = AutoTokenizer.from_pretrained(model_id)

 model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32).to(device)

 def pipe(
      prompt,
      max_new_tokens=100,         # maximum number of new tokens to generate (excluding the input prompt)
      temperature=0.0,            # controls randomness, with do_sample=False, generation is deterministic
      return_full_text=False,     # if True, return both the prompt and generated text, otherwise return only the generated response
      do_sample=False,            # if False, use greedy decoding, if True, sample from the token probability distribution
      clean_up_tokenization_spaces=False,  # whether to remove tokenization artefacts (e.g. extra spaces around punctuation)

  ):
      # prompt is expected to be:
      # [{"role":"user","content":"..."}]

      inputs = tokenizer.apply_chat_template(
          prompt,
          tokenize=True,
          add_generation_prompt=True,
          return_tensors="pt"
      )

      inputs = {k: v.to(model.device) for k, v in inputs.items()}

      outputs = model.generate(
          **inputs,
          max_new_tokens=max_new_tokens,
          temperature=temperature,
          do_sample=do_sample
      )

      input_length = inputs["input_ids"].shape[1]

      if return_full_text:
        generated_ids = outputs[0]
      else:
        generated_ids = outputs[0][input_length:]

      generated_text = tokenizer.decode(
          generated_ids,
          skip_special_tokens=True,
          clean_up_tokenization_spaces=clean_up_tokenization_spaces
      )

      return [{"generated_text": generated_text}]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

## ROUGE Metric

To evaluate our model's performance, we will use **ROUGE (Recall-Oriented Understudy for Gisting Evaluation)**. It compares *n*-grams of the model outputs to reference text and is based on recall.

ROUGE outputs multiple types of scores. ROUGE-*N* measures the overlap of *n*-grams between generated and reference summaries, with ROUGE-1 focusing on unigrams and ROUGE-2 on bigrams. ROUGE-L evaluates the longest common subsequence by identifying the longest co-occurring sequences of *n*-grams. ROUGE-Lsum focuses on the longest common subsequence at the summary level.


In [73]:
!pip install evaluate rouge_score

In [74]:
import evaluate
metric = evaluate.load("rouge", trust_remote_code=True)

# help(metric)      # << Uncomment to see more about the ROUGE eval metric

predicted_summaries = ["AI applications span diverse fields.", "Enhancing efficiency, decision-making, experiences with AI."]
target_doc = 'As artificial intelligence continues to advance, researchers are exploring its applications in diverse fields such as healthcare, finance, education, and entertainment, aiming to improve efficiency, decision-making processes, and overall human experiences.'
docs = [target_doc, target_doc]

metric.compute(predictions=predicted_summaries, references=docs)

{'rouge1': np.float64(0.18364518364518362),
 'rouge2': np.float64(0.08262548262548262),
 'rougeL': np.float64(0.18364518364518362),
 'rougeLsum': np.float64(0.18364518364518362)}

**Task 1**. Run the provided code to generate answers using the `MediPhi` model and analyse the baseline performance by comparing the generated answers to the reference answers.

In [75]:
generated_answers = []
reference_answers = []

for i, row in df.iterrows():
    question = row["question"]
    reference = row["answer"]
    messages = []

    messages.append({"role": "system", "content": "You are a friendly, knowledgeable medical expert who explains health topics in a clear, compassionate, and patient-friendly way."})
    messages.append({"role": "user", "content": question})


    # pipeline will take care of tokenisation and calling apply_chat_template for you
    generated = pipe(
        messages
    )[0]["generated_text"]

    print(f"Question: {question}")
    print(f"LLM Answer: {generated}")
    print(f"Reference Answer: {reference}")
    print('*******************')

    generated_answers.append(generated)
    reference_answers.append(reference)

results = metric.compute(predictions=generated_answers, references=reference_answers)
print("ROUGE Results:", results)

Question: Can I prevent my child from getting bronchiolitis?
LLM Answer: Bronchiolitis is a common respiratory infection in young children, typically caused by the respiratory syncytial virus (RSV). While it's not possible to completely prevent bronchiolitis, there are several strategies you can employ to reduce your child's risk of infection or to minimize the severity of the illness if they do get it:

1. Practice good hygiene: Teach your child to wash their
Reference Answer: You can lower the chances of your child getting bronchiolitis by washing your hands and your child's hands often, washing or wiping down toys and clean surfaces regularly, using disposable tissues, keeping newborn babies away from anyone with a cold or flu, and not smoking around your child.
*******************
Question: What are the early symptoms of bronchiolitis?
LLM Answer: Bronchiolitis is a common lung infection in young children and infants, primarily caused by the Respiratory Syncytial Virus (RSV). The e

## Retrieval-Augmented Generation (RAG) with TF-IDF

Recall that Retrieval-Augmented Generation (RAG) first retrieves relevant information from available data based on the input query and then uses this retrieved information as prompt context to generate more accurate responses.

In this part of the lab we will build a RAG Q&A system with TF-IDF vectors. We will start with helper functions.

**Before you proceed, create `search_data` folder in the lab space, open it and upload the `search_data` folder content provided with the lab to the opened folder.**

The `fetch_by_diagnosis` function uses the disease name to read data from specified directories.

In [76]:
search_data_dir = "./search_data"

# apply the method to match file naming convention
def safe_filename(name):
    return re.sub(r'[^\w\-_. ]', '_', name).strip().replace(" ", "_") + ".txt"


def fetch_by_diagnosis(diagnosis_name):
    filename = safe_filename(diagnosis_name)

    data_path = os.path.join(search_data_dir, filename)

    try:
        with open(data_path, "r", encoding="utf-8") as f:
            cleaned_text = f.read()
    except FileNotFoundError:
        cleaned_text = None

    return cleaned_text

**Task 2**: Complete the function `retrieve_with_tfidf` below. Use `TfidfVectorizer` from Scikit to vectorise a list of sentences and a query, and `NearestNeighbors` from Scikit to retrieve the top *k* most similar sentences to the query based on cosine similarity.

In [77]:
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer

def retrieve_with_tfidf(sentences, query, k=5):

    vectorizer = TfidfVectorizer()
    sentence_vectors = vectorizer.fit_transform(sentences)
    query_vector = vectorizer.transform([query])

    nn = NearestNeighbors(n_neighbors=k, metric='cosine')
    nn.fit(sentence_vectors)
    distances, indices = nn.kneighbors(query_vector)

    return [sentences[i] for i in indices[0]]

 Run the code below that retrieves relevant document parts using the TF-IDF method above, build a context prompt, and generate answers using the MediPhi model. Analyse the resulting performance.

In [78]:
generated_answers = []
reference_answers = []

for i, row in df.iterrows():
    question = row["question"]
    reference = row["answer"]
    disease = row["disease"]

    # retrieve relevant clinical notes using TF-IDF RAG
    related_contexts = retrieve_with_tfidf(
        fetch_by_diagnosis(disease).split('\n\n'),
        question, k=5
    )

    # build context prompt from the top-k relevant notes
    context_prompt = "\n\n".join(related_contexts)

    messages = [
        {"role": "system", "content": "You are a friendly medical expert who answers health-related topics in a clear and patient-friendly way."},
        {"role": "user", "content": f"Context:\n{context_prompt}\n\nQuestion:\n{question}"}
    ]

    generated = pipe(
        messages
    )[0]["generated_text"]

    print(f"Question: {question}")
    print(f"LLM Answer: {generated}")
    print(f"Reference Answer: {reference}")
    print('*******************')

    generated_answers.append(generated)
    reference_answers.append(reference)

# Compute metric
results = metric.compute(predictions=generated_answers, references=reference_answers)
print("ROUGE Results:", results)


Question: Can I prevent my child from getting bronchiolitis?
LLM Answer: Yes, there are several preventive measures you can take to lower the chances of your child getting bronchiolitis or spreading the viruses that cause it. Here are some tips:

1. Practice good hygiene: Wash your hands and your child's hands often, especially before meals and after using the toilet.
2. Clean surfaces and toys: Regularly wash or wipe down toys, surfaces, and objects
Reference Answer: You can lower the chances of your child getting bronchiolitis by washing your hands and your child's hands often, washing or wiping down toys and clean surfaces regularly, using disposable tissues, keeping newborn babies away from anyone with a cold or flu, and not smoking around your child.
*******************
Question: What are the early symptoms of bronchiolitis?
LLM Answer: The early symptoms of bronchiolitis are similar to those of a common cold. These include:

1. Sneezing
2. A runny or blocked nose
3. A cough
4. A 

## RAG with Glove Embeddings

Now we will use more advanced embedding and indexing techniques for data search.

For this, we will start by installing a couple of dependencies. We will use [Faiss](https://github.com/facebookresearch/faiss) (Facebook AI Similarity Search). It is a library developed by Facebook AI Research for efficient similarity search and clustering of dense vectors.

In [79]:
!pip install faiss-cpu

Now let's download the Glove embeddings.

In [80]:
# download the GloVe zip archive
!wget https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip --no-check-certificate
# unzip the file
!unzip glove.6B.zip

--2026-06-21 09:57:57--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glove.6B.zip        100%[===================>] 822.24M  5.06MB/s    in 2m 40s  

2026-06-21 10:00:38 (5.14 MB/s) - ‘glove.6B.zip’ saved [862182613/862182613]

Archive:  glove.6B.zip
  inflating: glove.6B.50d.txt        
  inflating: glove.6B.100d.txt       
  inflating: glove.6B.200d.txt       
  inflating: glove.6B.300d.txt       


**Task 3**. Complete the function to compute average word vectors to embed text.

In [81]:
# load gloVe embeddings
def load_glove(filepath="glove.6B.100d.txt"):
    glove = {}
    with open(filepath, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            word = parts[0]
            vec = np.array(parts[1:], dtype=np.float32)
            glove[word] = vec
    return glove

# average word vectors to embed text
def average_embed(text, glove, dim=100):
  words = text.lower().split()
  vectors = [glove[word] for word in words if word in glove]
  if len(vectors) == 0:
    return np.full(dim, 1e-8, dtype=np.float32)
  return np.mean(vectors, axis=0)

Run the code below that retrieves relevant document parts using the Glove method above. Analyse the resulting performance.

In [82]:
import faiss

glove = load_glove("glove.6B.100d.txt")

def get_chunks_with_faiss(chunk_texts, query):

    embeddings = np.array([average_embed(chunk, glove, 100) for chunk in chunk_texts])

    # initialise the FAISS index for performing similarity search using L2 (Euclidean) distance
    # 100 indicates the dimensionality of the vectors being indexed
    index = faiss.IndexFlatL2(100)
    index.add(embeddings)

    query_vec = average_embed(query, glove).reshape(1, -1)
    # we search in the FAISS index using the query vector
    # distances contain the distances of the retrieved vectors from the query vector
    # indices contain the indices of the retrieved vectors in the original embeddings
    distances, indices = index.search(query_vec, k=5)

    return [chunk_texts[i] for i in indices[0]]

unique_diagnoses = df['disease'].dropna().unique().tolist()

generated_answers = []
reference_answers = []

for i, row in df.iterrows():
    question = row["question"]
    reference = row["answer"]
    disease = row["disease"]

    # TO DO

    diagnosis_selection_prompt = [
        {"role": "system", "content": "You are a helpful assistant trained to identify relevant medical question topics."},
        {"role": "user", "content": f"Given the question below, which one of the following diagnoses is most relevant?\n\nDiagnoses:\n{', '.join(unique_diagnoses)}\n\nQuestion:\n{question}\n\nRespond with only the diagnosis name."}
    ]

    diagnosis_guess = pipe(diagnosis_selection_prompt, max_new_tokens=20)[0]["generated_text"].strip()

    # random fallback if diagnosis not found
    if diagnosis_guess not in unique_diagnoses:
        diagnosis_guess = random.choice(unique_diagnoses)

    print(f"Selected Diagnosis: {diagnosis_guess}")
    print('*******************')
    print(f"True Diagnosis: {disease}")
    print('*******************')

    disease = diagnosis_guess

    # TO DO

    doc_text_chunks = fetch_by_diagnosis(disease).split('\n\n')

    # retrieve relevant doc parts
    related_contexts = get_chunks_with_faiss(doc_text_chunks, question)

    context_prompt = "\n\n".join(related_contexts)
    messages = [
        {"role": "system", "content": "You are a friendly, knowledgeable medical expert who explains health topics in a clear, compassionate, and patient-friendly way."},
        {"role": "user", "content": f"Context:\n{context_prompt}\n\nQuestion:\n{question}"}
    ]

    generated = pipe(
        messages
    )[0]["generated_text"]

    print(f"Question: {question}")
    print(f"LLM Answer: {generated}")
    print(f"Reference Answer: {reference}")
    print('*******************\n\n')

    generated_answers.append(generated)
    reference_answers.append(reference)

results = metric.compute(predictions=generated_answers, references=reference_answers)
print("Evaluation Results:", results)


Selected Diagnosis: Bronchiolitis
*******************
True Diagnosis: Bronchiolitis
*******************
Question: Can I prevent my child from getting bronchiolitis?
LLM Answer: While it's not possible to completely prevent bronchiolitis, there are several measures you can take to reduce your child's risk of infection. Here are some tips:

1. Avoid exposure to secondhand smoke: Ensure that your child is not exposed to smoke, as it can irritate their airways and make them more susceptible to respiratory infections.

2. Practice good hygiene: Encourage frequent
Reference Answer: You can lower the chances of your child getting bronchiolitis by washing your hands and your child's hands often, washing or wiping down toys and clean surfaces regularly, using disposable tissues, keeping newborn babies away from anyone with a cold or flu, and not smoking around your child.
*******************


Selected Diagnosis: Bronchiolitis
*******************
True Diagnosis: Bronchiolitis
******************

**Task 4**: Write code to prompt MediPhi to select the most relevant diagnosis for a given medical question using the list of possible diagnoses.